# Finance ML Analytics Platform — v8_3

**Version 0.3.0** — Now using modular `finance_ml` package

## What's New

- All functions are now imported from the `finance_ml` package
- No need to define functions inline — they're maintained in the package modules
- Configuration management with `FinanceMLConfig`
- Better code organization and testability
- Feature flags for optional functionality control

## Modules

- `finance_ml.data`: Data loading, normalization, validation
- `finance_ml.features`: Feature engineering
- `finance_ml.models`: Classification, regression, ensembles
- `finance_ml.eval`: Analytics, visualizations, reporting
- `finance_ml.config`: Configuration management
- `finance_ml.cli`: Command-line interface

## Usage

This notebook demonstrates the ML workflow:
1. Load and validate data
2. Exploratory data analysis
3. Feature engineering
4. Model training (classification and regression)
5. Evaluation and analytics

## Configuration and Feature Flags

In [1]:
# Feature Availability via NotebookConfig
# Centralize feature flags using finance_ml.NotebookConfig; keep legacy variables for compatibility
from finance_ml import NotebookConfig

cfg = NotebookConfig(
        have_finance_prediction=True,
        have_database_connection=False,
        have_advanced_analytics=True,
        have_dim_reduction=False,
        debug_mode=False,
        enable_sector_analysis=True,
        enable_region_analysis=True,
        enable_interactive_plots=True,
        enable_excel_export=True,
        )

# Display a concise summary using the tested display API
cfg.display_summary()

# Backward-compatible variables (used later in the notebook)
HAVE_FINANCE_PREDICTION = cfg.have_finance_prediction
HAVE_DATABASE_CONNECTION = cfg.have_database_connection
HAVE_ADVANCED_ANALYTICS = cfg.have_advanced_analytics
HAVE_DIM_REDUCTION = cfg.have_dim_reduction
DEBUG_MODE = cfg.debug_mode
ENABLE_SECTOR_ANALYSIS = cfg.enable_sector_analysis
ENABLE_REGION_ANALYSIS = cfg.enable_region_analysis
ENABLE_INTERACTIVE_PLOTS = cfg.enable_interactive_plots
ENABLE_EXCEL_EXPORT = cfg.enable_excel_export

FEATURE FLAGS CONFIGURATION

Core Features:
  Financial Prediction:        ✓ Enabled
  Database Connection:         ✗ Disabled
  Advanced Analytics:          ✓ Enabled
  Dimensionality Reduction:    ✗ Disabled

Analysis Features:
  Sector Analysis:             ✓ Enabled
  Region Analysis:             ✓ Enabled

Output Features:
  Interactive Plots:           ✓ Enabled
  Excel Export:                ✓ Enabled

Development:
  Debug Mode:                  ✗ Disabled


In [2]:
# Finance ML Analytics Platform — Notebook (v0.3.0)
# This notebook now uses the modular finance_ml package

import warnings

warnings.filterwarnings('ignore')

# Data science libraries
import numpy as np
import pandas as pd

# Import all functions from finance_ml package
from finance_ml import (
    # Version
    __version__,
    # Configuration
    load_config,
    # Utilities
    setup_logging,
    # Data loading and validation
    # Notebook utilities
    display_config_summary,
    load_stock_data,
    display_data_summary,
    # Feature engineering
    # Modeling
    # Evaluation and analytics
    # Week 1 Enhancements - Data Quality & Monitoring
    )

# Setup logging
import logging

setup_logging()
logger = logging.getLogger(__name__)

print(f"Finance ML Analytics Platform v{__version__}")
print("All functions imported from finance_ml package")


Finance ML Analytics Platform v0.3.0
All functions imported from finance_ml package


## Configuration

Load configuration from environment variables or config files.


In [3]:
# Load configuration
config = load_config()
display_config_summary(config)


Configuration loaded:
  Data directory: data
  Output directory: outputs
  Model directory: models
  Cache directory: .cache
  Model version: v8_2
  Random seed: 42
  N jobs: -1
  Log level: INFO
  Memory limit: None
  DB URL: not configured


## Sample Data Generator

Create sample financial dataset for demonstration when real data is unavailable.

Note: The generator is now provided by the package as
`finance_ml.create_sample_financial_dataset` — no inline definition needed here.


## Data Loading

Load stock data from configured data source (database or CSV files) with automatic fallback to sample data.


In [4]:
# Load stock data using package strategy helpers
all_stocks = load_stock_data(config)
if all_stocks is None or len(all_stocks) == 0:
    raise ValueError("Failed to load any stock data")

display_data_summary(all_stocks)


2025-10-27 11:16:27,901 INFO Loading from CSV files...
2025-10-27 11:16:27,902 INFO Loading CSVs from data
2025-10-27 11:16:28,368 INFO ✓ Loaded 8000 stocks from CSV



Data Loading Summary
Loaded stocks: 8,000
Dataset shape: (8000, 230)
Columns (first 10): ['ticker', 'isin', 'name', 'description', 'exchange', 'unit', 'sector', 'industry', 'last_updated', 'income_statement_report_date']


## Data Validation and Quality Checks

Validate schema and check data quality using finance_ml package functions.


In [5]:
# Unified validation reporting with error handling
try:
    from finance_ml.data import validate_schema, check_missing_values

    # Schema validation
    try:
        validate_schema(all_stocks)
        print("✓ Schema validation passed")
    except Exception as e:
        print(f"⚠ Schema validation warning: {e}")

    # Missing values check
    try:
        missing_report = all_stocks.isnull().sum()
        missing_pct = (missing_report / len(all_stocks) * 100).round(2)
        missing_df = pd.DataFrame({
            'Missing Count': missing_report[missing_report > 0],
            'Missing %': missing_pct[missing_report > 0]
            }).sort_values('Missing Count', ascending=False)

        if len(missing_df) > 0:
            print("\n📊 Missing Values Report:")
            print(missing_df.head(10).to_string())
        else:
            print("✓ No missing values detected")
    except Exception as e:
        print(f"⚠ Missing value check failed: {e}")

except Exception as e:
    logger.error(f"Validation failed: {e}")
    print(f"⚠ Validation checks incomplete")

✓ Schema validation passed

📊 Missing Values Report:
                                    Missing Count  Missing %
impairment_of_goodwill_fq                    7861      98.26
interest_income_on_investments_ltm           7655      95.69
impairment_of_goodwill_fy                    7396      92.45
impairment_of_goodwill_ltm                   7326      91.57
avg_employees_ltm                            7183      89.79
restructuring_charges_fq                     7070      88.38
merger_restructuring_charges_fq              6739      84.24
restructuring_charges_fy                     6668      83.35
restructuring_charges_ltm                    6516      81.45
impairment_of_goodwill_5yavgfq               6297      78.71


## Exploratory Data Analysis

Perform EDA using the simple_eda function.


In [6]:
# Unified EDA display with proper output directory
try:
    from pathlib import Path

    output_dir = Path(config.output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    from finance_ml.eval import simple_eda

    print("\n" + "=" * 80)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 80)

    simple_eda(all_stocks, out_dir=output_dir)
    print(f"✓ EDA completed - outputs saved to {output_dir}")

except AttributeError as e:
    if "'DataFrame' object has no attribute 'dtype'" in str(e):
        logger.warning(f"EDA skipped due to dtype attribute error in simple_eda function: {e}")
        print(f"⚠ EDA skipped - known issue with simple_eda function")
        print(f"  Basic info: {all_stocks.shape[0]} rows, {all_stocks.shape[1]} columns")
    else:
        logger.error(f"EDA failed: {e}")
        print(f"⚠ EDA failed: {e}")
except Exception as e:
    logger.error(f"EDA failed: {e}")
    print(f"⚠ EDA failed: {e}")

2025-10-27 11:16:28,521 INFO Rows: 8000, Columns: 230



EXPLORATORY DATA ANALYSIS


2025-10-27 11:16:29,008 INFO Wrote EDA summary to outputs\eda_summary.json


✓ EDA completed - outputs saved to outputs


## Comprehensive Visualizations and Summary Statistics

Detailed analysis of the `all_stocks` dataframe with:
1. Summary statistics
2. Distribution visualizations
3. Correlation analysis
4. Interactive visualizations
5. Sector and region analysis
6. Financial metrics deep dive


In [7]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go

    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("⚠ Plotly not available - interactive plots will be skipped")

# Set visualization styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


### 1. Comprehensive Summary Statistics


In [8]:
# Dataset overview
print("=" * 80)
print("COMPREHENSIVE SUMMARY STATISTICS")
print("=" * 80)
print(f"\nDataset Shape: {all_stocks.shape[0]:,} rows × {all_stocks.shape[1]} columns")
print(f"Memory Usage: {all_stocks.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")

# Regional distribution
if 'region' in all_stocks.columns:
    print("\n📍 Regional Distribution:")
    region_counts = all_stocks['region'].value_counts()
    for region, count in region_counts.items():
        pct = (count / len(all_stocks) * 100)
        print(f"  {region}: {count:,} ({pct:.1f}%)")

# Sector distribution
if 'sector' in all_stocks.columns:
    print("\n🏢 Sector Distribution (Top 10):")
    sector_counts = all_stocks['sector'].value_counts().head(10)
    for sector, count in sector_counts.items():
        pct = (count / len(all_stocks) * 100)
        print(f"  {sector}: {count:,} ({pct:.1f}%)")

# Key financial metrics summary
numeric_cols = all_stocks.select_dtypes(include=[np.number]).columns.tolist()
key_metrics = [c for c in ['market_cap', 'last_price', 'p_e', 'ev_ebitda', 'revenue', 'net_income']
               if c in numeric_cols]

if key_metrics:
    print("\n💰 Key Financial Metrics Summary:")
    summary_stats = all_stocks[key_metrics].describe()
    print(summary_stats.to_string())

    # Data coverage
    print("\n📊 Data Coverage:")
    for col in key_metrics:
        coverage = (1 - all_stocks[col].isna().sum() / len(all_stocks)) * 100
        print(f"  {col}: {coverage:.1f}%")

# Target variable statistics
if 'price_target' in all_stocks.columns:
    print("\n🎯 Target Variable (price_target):")
    pt = all_stocks['price_target'].dropna()

    # Ensure pt is a Series (not DataFrame) and convert to numeric
    if len(pt) > 0:
        print(f"  Count: {len(pt):,}")

        # Robust handling to ensure 1-dimensional structure before pd.to_numeric()
        # Step 1: Handle DataFrame case
        if isinstance(pt, pd.DataFrame):
            if pt.shape[1] == 1:
                pt = pt.iloc[:, 0]
            else:
                # If multiple columns, try to select 'price_target' or use first column
                if 'price_target' in pt.columns:
                    pt = pt['price_target']
                else:
                    pt = pt.iloc[:, 0]
                    print(f"  Warning: Multiple columns found, using first column")

        # Step 2: Flatten if still multi-dimensional (numpy array case)
        if isinstance(pt, np.ndarray) and pt.ndim > 1:
            pt = pt.flatten()

        # Step 3: Convert to Series if it's a numpy array
        if not isinstance(pt, pd.Series):
            pt = pd.Series(pt)

        # Now safely convert to numeric, coercing errors to NaN
        pt_numeric = pd.to_numeric(pt, errors='coerce')

        # Remove NaN values
        pt_numeric = pt_numeric.dropna()

        if len(pt_numeric) > 0:
            print(f"  Mean: ${pt_numeric.mean():.2f}")
            print(f"  Median: ${pt_numeric.median():.2f}")
            print(f"  Std Dev: ${pt_numeric.std():.2f}")
        else:
            print("  No numeric values available for statistical analysis")

# Sector-wise aggregations
if 'sector' in all_stocks.columns and 'market_cap' in numeric_cols:
    print("\n🏆 Top 5 Sectors by Total Market Cap:")
    sector_mcap = all_stocks.groupby('sector')['market_cap'].sum().sort_values(ascending=False).head(5)
    for sector, mcap in sector_mcap.items():
        print(f"  {sector}: ${mcap / 1e9:.1f}B")


COMPREHENSIVE SUMMARY STATISTICS

Dataset Shape: 8,000 rows × 230 columns
Memory Usage: 30.48 MB

📍 Regional Distribution:
  Asia / Pacific: 2,053 (25.7%)
  Europe: 2,051 (25.6%)
  United States and Canada: 1,815 (22.7%)
  Africa / Middle East: 1,462 (18.3%)
  Latin America and Caribbean: 617 (7.7%)
  634.88: 1 (0.0%)

🏢 Sector Distribution (Top 10):
  Industrials: 1,409 (17.6%)
  Financials: 1,399 (17.5%)
  Consumer Discretionary: 877 (11.0%)
  Information Technology: 816 (10.2%)
  Materials: 715 (8.9%)
  Health Care: 625 (7.8%)
  Consumer Staples: 567 (7.1%)
  Real Estate: 553 (6.9%)
  Utilities: 359 (4.5%)
  Communication Services: 357 (4.5%)

💰 Key Financial Metrics Summary:
         market_cap    last_price          p_e        revenue     net_income
count  7.998000e+03  7.998000e+03  6722.000000    7927.000000    7988.000000
mean   1.693998e+04  4.039398e+03    30.868447    7861.612558     738.175401
std    1.059729e+05  4.355835e+04    42.312341   26063.683127    3684.612179
min 

### 2. Distribution Visualizations (Matplotlib/Seaborn)


In [9]:
# Regional distribution bar chart
if 'region' in all_stocks.columns:
    plt.figure(figsize=(10, 6))
    region_counts = all_stocks['region'].value_counts()
    sns.barplot(x=region_counts.index, y=region_counts.values, palette='Set2')
    plt.title('Stock Distribution by Region', fontsize=14, fontweight='bold')
    plt.xlabel('Region')
    plt.ylabel('Number of Stocks')
    for i, v in enumerate(region_counts.values):
        plt.text(i, v + len(all_stocks) * 0.01, str(v), ha='center', va='bottom')
    plt.tight_layout()
    plt.show()


In [10]:
# Top 10 sectors bar chart
if 'sector' in all_stocks.columns:
    plt.figure(figsize=(12, 6))
    sector_counts = all_stocks['sector'].value_counts().head(10)
    sns.barplot(x=sector_counts.values, y=sector_counts.index, palette='viridis')
    plt.title('Top 10 Sectors by Stock Count', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Stocks')
    plt.ylabel('Sector')
    for i, v in enumerate(sector_counts.values):
        plt.text(v + len(all_stocks) * 0.005, i, str(v), ha='left', va='center')
    plt.tight_layout()
    plt.show()


In [11]:
# Key financial metrics distributions with histograms and KDE
plot_metrics = [c for c in ['market_cap', 'last_price', 'p_e', 'revenue']
                if c in all_stocks.columns]

if plot_metrics:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, metric in enumerate(plot_metrics[:4]):
        data = all_stocks[metric].dropna()
        if len(data) > 0:
            # Use log scale for large-value metrics
            if metric in ['market_cap', 'revenue'] and data.max() > 1e6:
                data_plot = np.log10(data[data > 0])
                axes[idx].hist(data_plot, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
                axes[idx].set_xlabel(f'log10({metric})')
                axes[idx].set_title(f'Distribution of {metric} (log scale)', fontweight='bold')
            else:
                axes[idx].hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
                sns.kdeplot(data, ax=axes[idx], color='red', linewidth=2)
                axes[idx].set_xlabel(metric)
                axes[idx].set_title(f'Distribution of {metric}', fontweight='bold')

            axes[idx].set_ylabel('Frequency')
            axes[idx].axvline(data.mean() if metric not in ['market_cap', 'revenue'] else data_plot.mean(),
                              color='red', linestyle='--', linewidth=1, label=f'Mean: {data.mean():.2f}')
            axes[idx].legend()

    plt.tight_layout()
    plt.show()


In [12]:
# Box plots for key metrics by region
if 'region' in all_stocks.columns:
    box_metrics = [c for c in ['market_cap', 'p_e', 'last_price'] if c in all_stocks.columns]

    if box_metrics:
        fig, axes = plt.subplots(1, len(box_metrics), figsize=(15, 5))
        if len(box_metrics) == 1:
            axes = [axes]

        for idx, metric in enumerate(box_metrics):
            data_for_box = all_stocks[[metric, 'region']].dropna()
            if len(data_for_box) > 0:
                # Use log scale for market cap
                if metric == 'market_cap':
                    data_for_box[metric] = np.log10(data_for_box[metric].clip(lower=1))
                    axes[idx].set_ylabel(f'log10({metric})')
                else:
                    axes[idx].set_ylabel(metric)

                sns.boxplot(data=data_for_box, x='region', y=metric, palette='Set3', ax=axes[idx])
                axes[idx].set_title(f'{metric} by Region', fontweight='bold')
                axes[idx].set_xlabel('Region')

        plt.tight_layout()
        plt.show()


In [13]:
# Violin plots for P/E ratio by top 5 sectors
if 'sector' in all_stocks.columns and 'p_e' in all_stocks.columns:
    top_5_sectors = all_stocks['sector'].value_counts().head(5).index
    data_violin = all_stocks[all_stocks['sector'].isin(top_5_sectors)][['p_e', 'sector']].dropna()

    # Filter outliers for better visualization
    data_violin = data_violin[data_violin['p_e'].between(0, data_violin['p_e'].quantile(0.95))]

    if len(data_violin) > 0:
        plt.figure(figsize=(12, 6))
        sns.violinplot(data=data_violin, x='sector', y='p_e', palette='muted')
        plt.title('P/E Ratio Distribution by Top 5 Sectors', fontsize=14, fontweight='bold')
        plt.xlabel('Sector')
        plt.ylabel('P/E Ratio')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()


### 3. Correlation Analysis and Heatmaps


In [14]:
# Select numeric columns for correlation
numeric_cols = all_stocks.select_dtypes(include=[np.number]).columns.tolist()
corr_cols = [c for c in ['market_cap', 'last_price', 'p_e', 'p_b', 'ev_ebitda',
                         'revenue', 'net_income', 'ebitda', 'gross_margin', 'price_target']
             if c in numeric_cols]

if len(corr_cols) >= 3:
    corr_data = all_stocks[corr_cols].dropna()

    if len(corr_data) > 10:
        # Pearson correlation
        plt.figure(figsize=(12, 10))
        corr_pearson = corr_data.corr(method='pearson')
        sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                    square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Pearson Correlation Matrix (Key Financial Metrics)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Spearman correlation for non-linear relationships
        plt.figure(figsize=(12, 10))
        corr_spearman = corr_data.corr(method='spearman')
        sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='viridis', center=0,
                    square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Spearman Correlation Matrix (Key Financial Metrics)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # Top correlations summary
        print("\n🔗 Top 10 Positive Correlations:")
        corr_unstacked = corr_pearson.unstack()
        corr_unstacked = corr_unstacked[corr_unstacked < 1.0]  # Remove self-correlations
        top_pos = corr_unstacked.sort_values(ascending=False).head(10)
        for (var1, var2), val in top_pos.items():
            print(f"  {var1} ↔ {var2}: {val:.3f}")

        print("\n🔗 Top 10 Negative Correlations:")
        top_neg = corr_unstacked.sort_values(ascending=True).head(10)
        for (var1, var2), val in top_neg.items():
            print(f"  {var1} ↔ {var2}: {val:.3f}")



🔗 Top 10 Positive Correlations:
  last_price ↔ price_target: 0.974
  price_target ↔ last_price: 0.974
  ebitda ↔ net_income: 0.948
  net_income ↔ ebitda: 0.948
  market_cap ↔ net_income: 0.901
  net_income ↔ market_cap: 0.901
  ebitda ↔ market_cap: 0.813
  market_cap ↔ ebitda: 0.813
  ebitda ↔ revenue: 0.753
  revenue ↔ ebitda: 0.753

🔗 Top 10 Negative Correlations:
  gross_margin ↔ revenue: -0.135
  revenue ↔ gross_margin: -0.135
  revenue ↔ p_e: -0.054
  p_e ↔ revenue: -0.054
  net_income ↔ p_e: -0.052
  p_e ↔ net_income: -0.052
  ebitda ↔ p_e: -0.046
  p_e ↔ ebitda: -0.046
  gross_margin ↔ last_price: -0.025
  last_price ↔ gross_margin: -0.025


In [15]:
# Pair plot for key metrics (sample for performance)
pair_metrics = [c for c in ['market_cap', 'last_price', 'p_e', 'revenue']
                if c in all_stocks.columns]

if len(pair_metrics) >= 3 and 'sector' in all_stocks.columns:
    # Sample data for performance
    sample_size = min(500, len(all_stocks))
    sample_data = all_stocks[pair_metrics + ['sector']].dropna().sample(n=sample_size, random_state=42)

    # Use log scale for large metrics
    for col in ['market_cap', 'revenue']:
        if col in sample_data.columns:
            sample_data[col] = np.log10(sample_data[col].clip(lower=1))
            sample_data.rename(columns={col: f'log10_{col}'}, inplace=True)

    print(f"Generating pair plot with {len(sample_data)} samples...")
    sns.pairplot(sample_data, hue='sector', diag_kind='kde', corner=True, palette='Set2')
    plt.suptitle('Pair Plot: Key Financial Metrics', y=1.01, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


Generating pair plot with 500 samples...


In [16]:
# Feature correlation with price_target
if 'price_target' in all_stocks.columns:
    target_corr_cols = [c for c in numeric_cols if c != 'price_target'][:15]  # Top 15 features

    if target_corr_cols:
        target_corr_data = all_stocks[target_corr_cols + ['price_target']].dropna()

        if len(target_corr_data) > 10:
            correlations = target_corr_data.corr()['price_target'].drop('price_target').sort_values(ascending=False)

            plt.figure(figsize=(10, 8))
            colors = ['green' if x > 0 else 'red' for x in correlations.values]
            plt.barh(range(len(correlations)), correlations.values, color=colors, alpha=0.7)
            plt.yticks(range(len(correlations)), correlations.index)
            plt.xlabel('Correlation with price_target')
            plt.title('Feature Correlation with Price Target', fontsize=14, fontweight='bold')
            plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
            plt.tight_layout()
            plt.show()


### 4. Interactive Visualizations (Plotly)


In [17]:
# Interactive scatter: Market Cap vs P/E by sector
if HAVE_PLOTLY and 'market_cap' in all_stocks.columns and 'p_e' in all_stocks.columns:
    scatter_data = all_stocks[['market_cap', 'p_e', 'sector', 'ticker', 'region']].dropna()
    # Filter outliers for better visualization
    scatter_data = scatter_data[
        (scatter_data['p_e'] > 0) &
        (scatter_data['p_e'] < scatter_data['p_e'].quantile(0.95)) &
        (scatter_data['market_cap'] > 0)
        ]

    if len(scatter_data) > 0:
        fig = px.scatter(
                scatter_data,
                x='market_cap',
                y='p_e',
                color='sector',
                hover_data=['ticker', 'region'],
                log_x=True,
                title='Market Cap vs P/E Ratio by Sector (Interactive)',
                labels={'market_cap': 'Market Cap (log scale)', 'p_e': 'P/E Ratio'},
                height=600
                )
        fig.update_layout(showlegend=True)
        fig.show()


In [18]:
# Sunburst chart: Region → Sector hierarchy
if HAVE_PLOTLY and 'region' in all_stocks.columns and 'sector' in all_stocks.columns:
    sunburst_data = all_stocks[['region', 'sector']].dropna()
    sunburst_counts = sunburst_data.groupby(['region', 'sector']).size().reset_index(name='count')

    if len(sunburst_counts) > 0:
        fig = px.sunburst(
                sunburst_counts,
                path=['region', 'sector'],
                values='count',
                title='Stock Distribution: Region → Sector Hierarchy',
                height=700
                )
        fig.update_traces(textinfo='label+percent parent')
        fig.show()


In [19]:
# Interactive box plot: Valuation metrics by region
if HAVE_PLOTLY and 'region' in all_stocks.columns:
    box_metric = 'p_e' if 'p_e' in all_stocks.columns else 'last_price'
    box_data = all_stocks[[box_metric, 'region']].dropna()

    # Filter outliers
    box_data = box_data[box_data[box_metric].between(
            box_data[box_metric].quantile(0.05),
            box_data[box_metric].quantile(0.95)
            )]

    if len(box_data) > 0:
        fig = px.box(
                box_data,
                x='region',
                y=box_metric,
                color='region',
                title=f'{box_metric.upper()} Distribution by Region (Interactive)',
                labels={box_metric: box_metric.upper(), 'region': 'Region'},
                height=500
                )
        fig.update_layout(showlegend=False)
        fig.show()


In [20]:
# Bar chart: Top 20 stocks by market cap
if HAVE_PLOTLY and 'market_cap' in all_stocks.columns and 'ticker' in all_stocks.columns:
    top_stocks = all_stocks.nlargest(20, 'market_cap')[['ticker', 'market_cap', 'sector']].copy()

    if len(top_stocks) > 0:
        fig = px.bar(
                top_stocks,
                x='ticker',
                y='market_cap',
                color='sector',
                title='Top 20 Stocks by Market Cap',
                labels={'market_cap': 'Market Cap ($)', 'ticker': 'Ticker'},
                height=500
                )
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()


In [21]:
# 3D scatter: Market Cap, P/E, and Revenue Growth
if HAVE_PLOTLY:
    scatter_3d_cols = ['market_cap', 'p_e', 'revenue']
    scatter_3d_available = all([c in all_stocks.columns for c in scatter_3d_cols])

    if scatter_3d_available and 'sector' in all_stocks.columns:
        scatter_3d_data = all_stocks[scatter_3d_cols + ['sector', 'ticker']].dropna()

        # Filter outliers
        for col in scatter_3d_cols:
            scatter_3d_data = scatter_3d_data[
                scatter_3d_data[col].between(
                        scatter_3d_data[col].quantile(0.05),
                        scatter_3d_data[col].quantile(0.95)
                        )
            ]

        if len(scatter_3d_data) > 10:
            fig = px.scatter_3d(
                    scatter_3d_data,
                    x='market_cap',
                    y='p_e',
                    z='revenue',
                    color='sector',
                    hover_data=['ticker'],
                    log_x=True,
                    log_z=True,
                    title='3D View: Market Cap, P/E, and Revenue by Sector',
                    labels={
                        'market_cap': 'Market Cap (log)',
                        'p_e': 'P/E Ratio',
                        'revenue': 'Revenue (log)'
                        },
                    height=700
                    )
            fig.show()


In [22]:
# Treemap: Market cap distribution by region and sector
if HAVE_PLOTLY and 'region' in all_stocks.columns and 'sector' in all_stocks.columns and 'market_cap' in all_stocks.columns:
    treemap_data = all_stocks[['region', 'sector', 'market_cap']].dropna()
    treemap_agg = treemap_data.groupby(['region', 'sector'])['market_cap'].sum().reset_index()

    if len(treemap_agg) > 0:
        fig = px.treemap(
                treemap_agg,
                path=['region', 'sector'],
                values='market_cap',
                title='Market Cap Distribution by Region and Sector',
                height=700
                )
        fig.update_traces(textinfo='label+value+percent parent')
        fig.show()


In [23]:
# Grouped bar chart: Current price vs predicted target (if available)
if HAVE_PLOTLY and 'last_price' in all_stocks.columns and 'price_target' in all_stocks.columns:
    comparison_data = all_stocks[['ticker', 'last_price', 'price_target', 'sector']].dropna()

    # Take top 20 stocks by market cap if available, else first 20
    if 'market_cap' in all_stocks.columns:
        top_tickers = all_stocks.nlargest(20, 'market_cap')['ticker'].tolist()
        comparison_data = comparison_data[comparison_data['ticker'].isin(top_tickers)]
    else:
        comparison_data = comparison_data.head(20)

    if len(comparison_data) > 0:
        fig = go.Figure()

        fig.add_trace(go.Bar(
                x=comparison_data['ticker'],
                y=comparison_data['last_price'],
                name='Current Price',
                marker_color='steelblue'
                ))

        fig.add_trace(go.Bar(
                x=comparison_data['ticker'],
                y=comparison_data['price_target'],
                name='Price Target',
                marker_color='orange'
                ))

        fig.update_layout(
                title='Current Price vs Price Target (Top Stocks)',
                xaxis_title='Ticker',
                yaxis_title='Price ($)',
                barmode='group',
                height=500,
                xaxis_tickangle=-45
                )

        fig.show()


### 5. Sector and Region Analysis


In [24]:
# Sector-Region market cap heatmap
if 'sector' in all_stocks.columns and 'region' in all_stocks.columns and 'market_cap' in all_stocks.columns:
    heatmap_data = all_stocks.groupby(['sector', 'region'])['market_cap'].sum().unstack(fill_value=0)

    # Get top 10 sectors by total market cap
    top_sectors = heatmap_data.sum(axis=1).nlargest(10).index
    heatmap_data = heatmap_data.loc[top_sectors]

    if not heatmap_data.empty:
        plt.figure(figsize=(10, 8))
        sns.heatmap(heatmap_data / 1e9, annot=True, fmt='.1f', cmap='YlOrRd',
                    linewidths=0.5, cbar_kws={'label': 'Market Cap ($B)'})
        plt.title('Market Cap Distribution: Top 10 Sectors by Region', fontsize=14, fontweight='bold')
        plt.xlabel('Region')
        plt.ylabel('Sector')
        plt.tight_layout()
        plt.show()


In [25]:
# Average valuation metrics by sector
if 'sector' in all_stocks.columns:
    valuation_metrics = [c for c in ['p_e', 'p_b', 'ev_ebitda'] if c in all_stocks.columns]

    if valuation_metrics:
        sector_valuations = all_stocks.groupby('sector')[valuation_metrics].mean().sort_values(
                by=valuation_metrics[0], ascending=False
                ).head(10)

        if not sector_valuations.empty:
            sector_valuations.plot(kind='barh', figsize=(12, 8), width=0.8)
            plt.title('Average Valuation Metrics by Sector (Top 10)', fontsize=14, fontweight='bold')
            plt.xlabel('Average Value')
            plt.ylabel('Sector')
            plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()


In [26]:
# Regional stock count and average market cap
if 'region' in all_stocks.columns and 'market_cap' in all_stocks.columns:
    regional_stats = all_stocks.groupby('region').agg({
        'ticker': 'count',
        'market_cap': 'mean'
        }).rename(columns={'ticker': 'stock_count', 'market_cap': 'avg_market_cap'})

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Stock count by region
    regional_stats['stock_count'].plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Stock Count by Region', fontweight='bold')
    axes[0].set_xlabel('Region')
    axes[0].set_ylabel('Number of Stocks')
    axes[0].tick_params(axis='x', rotation=45)

    # Average market cap by region
    (regional_stats['avg_market_cap'] / 1e9).plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
    axes[1].set_title('Average Market Cap by Region', fontweight='bold')
    axes[1].set_xlabel('Region')
    axes[1].set_ylabel('Avg Market Cap ($B)')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()


In [27]:
# Top 5 sectors distribution across regions (interactive)
has_required_columns = (
        'sector' in all_stocks.columns and
        'region' in all_stocks.columns
)

if HAVE_PLOTLY and has_required_columns:
    sector_counts = all_stocks['sector'].value_counts()
    top_5_sectors = sector_counts.head(5).index
    sector_region_data = all_stocks[all_stocks['sector'].isin(top_5_sectors)]